In [11]:
from pathlib import Path
from pprint import pprint

import mosaic

print("Mosaic version:", mosaic.__version__)
print("Number of public API exports:", len(mosaic.__all__))
print("Some public APIs:")
pprint(mosaic.__all__[:20])

Mosaic version: 0.1.0
Number of public API exports: 63
Some public APIs:
['AgentQueryKind',
 'BackendCapabilities',
 'BenchmarkArtifact',
 'CameraIntrinsics',
 'CameraMount',
 'CameraObservation',
 'CameraRig',
 'ClaimStatus',
 'DeterministicSplatBackend',
 'Embodiment',
 'EntityKind',
 'EvaluationReadinessResult',
 'EvidenceReference',
 'FailureClass',
 'FrameId',
 'FusionInput',
 'FusionResult',
 'FusionStatus',
 'GaussianSplat',
 'ImageBuffer']


In [12]:
from mosaic.platforms import default_platform_registry

registry = default_platform_registry()

print("Built-in platforms:", registry.names)

for name in registry.names:
    platform = registry.get(name)
    print()
    print("Platform:", platform.name)
    print("Robot type:", platform.embodiment.value)
    print("Base frame:", platform.base_frame.value)
    print("Cameras:", platform.camera_rig.camera_ids)


Built-in platforms: ('humanoid-stereo', 'quadruped-front', 'ugv-surround')

Platform: humanoid-stereo
Robot type: humanoid
Base frame: base_link
Cameras: ('left_eye', 'right_eye')

Platform: quadruped-front
Robot type: quadruped
Base frame: base_link
Cameras: ('front',)

Platform: ugv-surround
Robot type: ugv
Base frame: base_link
Cameras: ('front', 'left', 'right', 'rear')


In [13]:
from mosaic.evaluation import synthetic_forward_sequence

scenario = synthetic_forward_sequence(frame_count=5)

print("Scenario name:", scenario.name)
print("Number of frames:", len(scenario.sequence))
print("Camera IDs in first frame:", scenario.sequence.observations[0].camera_ids)

first_observation = scenario.sequence.observations[0]
print("First timestamp:", first_observation.timestamp.nanoseconds, "nanoseconds")
print("First camera pose:", first_observation.views[0].camera_pose)


Scenario name: synthetic-forward-5
Number of frames: 5
Camera IDs in first frame: ('front',)
First timestamp: 0 nanoseconds
First camera pose: Pose3(parent_frame=FrameId(value='world'), child_frame=FrameId(value='camera/front'), translation=Translation3(x=0.0, y=0.0, z=0.0), rotation=Quaternion(w=1.0, x=0.0, y=0.0, z=0.0))


In [14]:
from mosaic.core import FrameId
from mosaic.slam import DeterministicSplatBackend

backend = DeterministicSplatBackend(map_frame=FrameId("map"))
updates = []

for observation in scenario.sequence:
    update = backend.process(observation)
    updates.append(update)
    print(
        "time:",
        observation.timestamp.nanoseconds,
        "| tracking:",
        update.tracking_state.value,
        "| new splats:",
        len(update.map_update),
        "| total splats:",
        update.diagnostics["splat_count"],
    )

snapshot = backend.snapshot()

print()
print("Final map frame:", snapshot.frame.value)
print("Total splats in map:", len(snapshot))
print("First splat:", snapshot.splats[0])


time: 0 | tracking: tracking | new splats: 1 | total splats: 1
time: 100000000 | tracking: tracking | new splats: 1 | total splats: 2
time: 200000000 | tracking: tracking | new splats: 1 | total splats: 3
time: 300000000 | tracking: tracking | new splats: 1 | total splats: 4
time: 400000000 | tracking: tracking | new splats: 1 | total splats: 5

Final map frame: map
Total splats in map: 5
First splat: GaussianSplat(center=(0.0, 0.0, 1.0), covariance_upper=(0.01, 0.0, 0.0, 0.01, 0.0, 0.01), color_rgb=(1.0, 1.0, 1.0), opacity=1.0, provenance='front')


In [15]:
from mosaic.evaluation import run_slam_benchmark

artifact = run_slam_benchmark(
    scenario=scenario,
    backend=DeterministicSplatBackend(map_frame=FrameId("map")),
)

pprint(artifact.as_dict())

{'backend_metadata': {'name': 'deterministic-splat',
                      'parameter_deterministic': True,
                      'runs_out_of_process': False,
                      'supports_incremental_mapping': True,
                      'supports_multi_view': True,
                      'supports_uncertainty': False},
 'backend_name': 'deterministic-splat',
 'diagnostics': {'last_splat_count': 5,
                 'metric_lost_tracking_count': 0.0,
                 'metric_not_initialized_tracking_count': 0.0,
                 'metric_splat_count': 5.0,
                 'metric_update_count': 5.0,
                 'snapshot_splat_count': 5},
 'failure_classification': 'none',
 'gate_results': [],
 'metrics': [{'family': 'lifecycle',
              'name': 'update_count',
              'unit': 'count',
              'value': 5.0},
             {'family': 'lifecycle',
              'name': 'tracking_success_rate',
              'unit': 'ratio',
              'value': 1.0},
           

In [16]:
from mosaic.evaluation import FailureClass, RegressionGate, check_regression_gates

gates = (
    RegressionGate(
        metric_name="tracking_success_rate",
        minimum=1.0,
        failure_class=FailureClass.LIFECYCLE,
    ),
    RegressionGate(
        metric_name="splat_count",
        minimum=5.0,
        failure_class=FailureClass.MAP_FUSION,
    ),
)

results = check_regression_gates(artifact, gates)

for result in results:
    pprint(result.as_dict())

{'failure_class': 'lifecycle',
 'metric_name': 'tracking_success_rate',
 'observed': 1.0,
 'passed': True,
 'threshold': 1.0}
{'failure_class': 'map_fusion',
 'metric_name': 'splat_count',
 'observed': 5.0,
 'passed': True,
 'threshold': 5.0}


In [17]:
from mosaic.fusion import FusionInput, ProvenancePreservingFusion

backend_a = DeterministicSplatBackend(map_frame=FrameId("map"))
backend_b = DeterministicSplatBackend(map_frame=FrameId("map"))

for observation in synthetic_forward_sequence(frame_count=3).sequence:
    backend_a.process(observation)

for observation in synthetic_forward_sequence(frame_count=2).sequence:
    backend_b.process(observation)

map_a = backend_a.snapshot()
map_b = backend_b.snapshot()

fusion = ProvenancePreservingFusion()
fusion_result = fusion.fuse(FusionInput(maps=(map_a, map_b)))

print("Fusion strategy:", fusion.name)
print("Fusion status:", fusion_result.status.value)
print("Input map A splats:", len(map_a))
print("Input map B splats:", len(map_b))
print("Fused map splats:", len(fusion_result.fused_map))

print()
print("Provenance counts:")
pprint(dict(fusion_result.provenance.per_provenance_counts))

print()
print("Diagnostics:")
pprint(dict(fusion_result.diagnostics))

Fusion strategy: provenance-preserving
Fusion status: fused
Input map A splats: 3
Input map B splats: 2
Fused map splats: 5

Provenance counts:
{'front': 5}

Diagnostics:
{'fused_splat_count': 5,
 'fusion_conflict_candidate_count': 0,
 'fusion_uncertainty_model': 'preserved_not_optimized',
 'source_map_count': 2,
 'source_splat_count': 5}


In [18]:
from mosaic.agents import LocalRadiusQuery, SplatPosteriorView

view = SplatPosteriorView(fusion_result.fused_map)

print("Posterior summary:")
print(view.summary())

print()
print("Uncertainty summary:")
print(view.uncertainty())

print()
print("Provenance summary:")
pprint(dict(view.provenance().counts))

print()
print("Local geometry query near x=0:")
query = LocalRadiusQuery(
    center=(0.0, 0.0, 1.0),
    radius=0.25,
    max_results=10,
)

query_result = view.query_local(query)
print(query_result)
print("Matching splat IDs:", query_result.splat_ids)


Posterior summary:
PosteriorSummary(splat_count=5, provenance_count=1, mean_opacity=1.0)

Uncertainty summary:
PosteriorUncertaintySummary(splat_count=5, mean_covariance_diagonal=(0.01, 0.01, 0.01), max_covariance_diagonal=(0.01, 0.01, 0.01), mean_opacity=1.0, min_opacity=1.0, max_opacity=1.0, semantics='descriptive_only')

Provenance summary:
{'front': 5}

Local geometry query near x=0:
SplatQueryResult(query_kind=<AgentQueryKind.LOCAL_GEOMETRY: 'local_geometry'>, splat_ids=(0, 3), total_matches=2, truncated=False)
Matching splat IDs: (0, 3)


In [19]:
from mosaic.evaluation import write_benchmark_artifact

output_path = Path("mosaic-demo-benchmark.json")
write_benchmark_artifact(artifact, output_path)

print("Wrote:", output_path.resolve())
print(output_path.read_text())

Wrote: /home/prannayag/pragnition/robotics/mosaic/mosaic-demo-benchmark.json
{
  "backend_metadata": {
    "name": "deterministic-splat",
    "parameter_deterministic": true,
    "runs_out_of_process": false,
    "supports_incremental_mapping": true,
    "supports_multi_view": true,
    "supports_uncertainty": false
  },
  "backend_name": "deterministic-splat",
  "diagnostics": {
    "last_splat_count": 5,
    "metric_lost_tracking_count": 0.0,
    "metric_not_initialized_tracking_count": 0.0,
    "metric_splat_count": 5.0,
    "metric_update_count": 5.0,
    "snapshot_splat_count": 5
  },
  "failure_classification": "none",
  "gate_results": [],
  "metrics": [
    {
      "family": "lifecycle",
      "name": "update_count",
      "unit": "count",
      "value": 5.0
    },
    {
      "family": "lifecycle",
      "name": "tracking_success_rate",
      "unit": "ratio",
      "value": 1.0
    },
    {
      "family": "lifecycle",
      "name": "lost_tracking_count",
      "unit": "count"